# 05. 멀티모달 인덱싱 — PDF/PPTX 업로드 및 인덱서 실행

이 노트북에서는:
1. 로컬 PDF/PPTX 파일을 Blob Storage에 업로드합니다.
2. 사전 등록된 2개의 인덱서를 실행하여 각각 다른 방식으로 인덱싱합니다.

## 두 가지 멀티모달 파이프라인

| 파이프라인 | 인덱서 | 스킬 체인 | Function App |
|-----------|--------|-----------|-------------|
| **[B] Basic** | `st-multimodal-basic-indexer` | DI Layout (Cross-Region EUS2) → Native SplitSkill (markdown mode) → Embedding → Index | ❌ 불필요 |
| **[C] Verbalized** | `st-multimodal-verbalized-indexer` | DI Layout (EUS2) → Custom WebApiSkill (GPT-5.4 Verbalize) → Custom WebApiSkill (markdown_split) → Embedding → Index | ✅ skills-function |

## 사전 요구사항
- `01-infra-deployment.ipynb` 실행 완료 (인덱서 사전 등록 포함)
- `.env` 파일 생성 완료
- Blob Storage Private Endpoint 승인 완료

## 데이터 흐름
```
[로컬 PDF/PPTX]
    │  업로드
    ▼
[Blob Storage — raw-documents/raw/pdf/st/]
    │  AI Search Indexer 실행
    ├────────────────────────────────────────┐
    ▼ [B] Basic                              ▼ [C] Verbalized
[DI Layout → SplitSkill → Embedding]   [DI Layout → GPT-5.4 → Split → Embedding]
    │                                        │
    ▼                                        ▼
[st-multimodal-basic-index]             [st-multimodal-verbalized-index]
```

## 1. 환경 설정

In [22]:
import os
import sys
import json
import subprocess
import time
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

# 필수 환경변수 확인
required_vars = [
    'AZURE_SEARCH_SERVICE_ENDPOINT',
    'AZURE_STORAGE_ACCOUNT_NAME',
    'AZURE_STORAGE_RESOURCE_ID',
    'AZURE_RESOURCE_GROUP',
]
for var in required_vars:
    assert os.environ.get(var), f'{var} 환경변수가 설정되지 않았습니다.'

STORAGE_NAME = os.environ['AZURE_STORAGE_ACCOUNT_NAME']
SEARCH_ENDPOINT = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
RG_NAME = os.environ['AZURE_RESOURCE_GROUP']

# 인덱서/인덱스 이름 (setup_ai_search_multimodal_pipeline.py 와 동일)
SOURCE = 'st'  # source prefix
PDF_INDEXER = f'{SOURCE}-multimodal-pdf-indexer'           # [B-PDF] basic
PPTX_INDEXER = f'{SOURCE}-multimodal-pptx-indexer'         # [B-PPTX] basic
VERBALIZED_INDEXER = f'{SOURCE}-multimodal-verbalized-indexer'  # [C] verbalized
PDF_INDEX = f'{SOURCE}-multimodal-pdf-index'
PPTX_INDEX = f'{SOURCE}-multimodal-pptx-index'
VERBALIZED_INDEX = f'{SOURCE}-multimodal-verbalized-index'

# Blob 경로 (setup script datasource prefix='raw/' → pdf/pptx 서브폴더 모두 감시)
CONTAINER_NAME = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')
BLOB_PREFIX_PDF = f'raw/pdf/{SOURCE}/'
BLOB_PREFIX_PPTX = f'raw/pptx/{SOURCE}/'

print('환경 설정 완료')
print(f'  Storage   : {STORAGE_NAME}')
print(f'  AI Search : {SEARCH_ENDPOINT}')
print(f'  Container : {CONTAINER_NAME}')
print(f'    PDF     : {BLOB_PREFIX_PDF}')
print(f'    PPTX    : {BLOB_PREFIX_PPTX}')
print(f'  Indexers  : {PDF_INDEXER}, {PPTX_INDEXER}, {VERBALIZED_INDEXER}')


환경 설정 완료
  Storage   : stragidyn6dtfun6
  AI Search : https://search-ragi-dyn6dtfu.search.windows.net
  Container : raw-documents
    PDF     : raw/pdf/st/
    PPTX    : raw/pptx/st/
  Indexers  : st-multimodal-pdf-indexer, st-multimodal-pptx-indexer, st-multimodal-verbalized-indexer


## 2. PDF/PPTX 파일 업로드

로컬 파일을 Blob Storage `raw-documents/raw/pdf/st/` 경로에 업로드합니다.  
AI Search Indexer의 Data Source가 이 경로를 감시하고 있습니다.

In [2]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()
blob_service = BlobServiceClient(
    account_url=f'https://{STORAGE_NAME}.blob.core.windows.net',
    credential=credential,
)

container_client = blob_service.get_container_client(CONTAINER_NAME)

# 컨테이너 존재 확인
try:
    container_client.get_container_properties()
    print(f'✅ 컨테이너 확인: {CONTAINER_NAME}')
except Exception:
    container_client.create_container()
    print(f'컨테이너 생성: {CONTAINER_NAME}')

✅ 컨테이너 확인: raw-documents


In [3]:
# ── 업로드할 파일 경로 지정 ─────────────────────────────────────
# 디렉토리를 지정하면 하위 .pdf / .pptx 파일 전체를 업로드합니다.
PDF_SOURCE = '../data/raw'   # ← 여기에 로컬 파일 경로 입력

SUPPORTED_EXTENSIONS = {'.pdf', '.pptx'}

source_path = Path(PDF_SOURCE)
if source_path.is_dir():
    local_files = sorted(
        f for f in source_path.rglob('*')
        if f.suffix.lower() in SUPPORTED_EXTENSIONS and f.is_file()
    )
elif source_path.is_file() and source_path.suffix.lower() in SUPPORTED_EXTENSIONS:
    local_files = [source_path]
else:
    local_files = []

print(f'업로드 대상: {len(local_files)}개 파일')
for f in local_files:
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name} ({size_kb:,} KB)')

업로드 대상: 5개 파일
  ST_0008_0001055.pdf (555 KB)
  ST_0008_0001055.pptx (576 KB)
  ST_0008_0001104.pdf (476 KB)
  ST_0008_0010802.pptx (272 KB)
  ST_0028_0008746.pdf (714 KB)


In [4]:
# ── Blob Storage에 업로드 (확장자별 폴더 분리) ─────────────────────────────────────
uploaded = []
for file_path in local_files:
    ext = file_path.suffix.lower()
    prefix = BLOB_PREFIX_PDF if ext == '.pdf' else BLOB_PREFIX_PPTX
    blob_name = f'{prefix}{file_path.name}'
    with open(file_path, 'rb') as f:
        container_client.upload_blob(name=blob_name, data=f, overwrite=True)
    size_kb = file_path.stat().st_size // 1024
    print(f'  ✓ {blob_name} ({size_kb:,} KB)')
    uploaded.append(blob_name)

print(f'\n총 {len(uploaded)}개 파일 업로드 완료')


  ✓ raw/pdf/st/ST_0008_0001055.pdf (555 KB)
  ✓ raw/pptx/st/ST_0008_0001055.pptx (576 KB)
  ✓ raw/pdf/st/ST_0008_0001104.pdf (476 KB)
  ✓ raw/pptx/st/ST_0008_0010802.pptx (272 KB)
  ✓ raw/pdf/st/ST_0028_0008746.pdf (714 KB)

총 5개 파일 업로드 완료


In [5]:
# ── 컨테이너 내 업로드된 파일 확인 (PDF + PPTX) ────────────────────────────
for prefix in [BLOB_PREFIX_PDF, BLOB_PREFIX_PPTX]:
    blobs = list(container_client.list_blobs(name_starts_with=prefix))
    print(f'{CONTAINER_NAME}/{prefix} — {len(blobs)}개 파일')
    for b in blobs:
        print(f'  {b.name}  ({b.size:,} bytes)')


raw-documents/raw/pdf/st/ — 3개 파일
  raw/pdf/st/ST_0008_0001055.pdf  (568,583 bytes)
  raw/pdf/st/ST_0008_0001104.pdf  (487,957 bytes)
  raw/pdf/st/ST_0028_0008746.pdf  (731,457 bytes)
raw-documents/raw/pptx/st/ — 2개 파일
  raw/pptx/st/ST_0008_0001055.pptx  (590,351 bytes)
  raw/pptx/st/ST_0008_0010802.pptx  (278,601 bytes)


## 3. 사전 등록된 인덱서 확인

노트북 01에서 등록한 2개의 멀티모달 인덱서가 존재하는지 확인합니다.

In [6]:
import requests

API_VERSION = '2024-11-01-preview'

def get_search_headers():
    token = credential.get_token('https://search.azure.com/.default').token
    return {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

def check_indexer(indexer_name: str) -> dict | None:
    """인덱서 존재 여부 확인"""
    url = f'{SEARCH_ENDPOINT}/indexers/{indexer_name}?api-version={API_VERSION}'
    resp = requests.get(url, headers=get_search_headers())
    if resp.status_code == 200:
        return resp.json()
    return None

# 인덱서 확인
INDEXERS = [PDF_INDEXER, PPTX_INDEXER, VERBALIZED_INDEXER]
for name in INDEXERS:
    indexer = check_indexer(name)
    if indexer:
        print(f'✅ {name} — 등록됨 (skillset: {indexer.get("skillsetName", "N/A")})')
    else:
        print(f'❌ {name} — 미등록! scripts/setup_ai_search_multimodal_pipeline.py 실행 필요')


✅ st-multimodal-pdf-indexer — 등록됨 (skillset: st-multimodal-pdf-skillset)
✅ st-multimodal-pptx-indexer — 등록됨 (skillset: st-multimodal-pptx-skillset)
✅ st-multimodal-verbalized-indexer — 등록됨 (skillset: st-multimodal-verbalized-skillset)


## 4. 인덱서 실행

두 인덱서를 순차적으로 실행합니다.  
- **[B] Basic**: DI Layout → SplitSkill (markdown) → Embedding (빠름, ~2-5분/PDF)
- **[C] Verbalized**: DI Layout → GPT-5.4 Verbalize → Split → Embedding (느림, ~5-15분/PDF)

> Basic 인덱서를 먼저 실행하고 완료 후 Verbalized를 실행합니다 (리소스 경합 방지).

In [7]:
def run_indexer(indexer_name: str) -> bool:
    """인덱서 실행"""
    url = f'{SEARCH_ENDPOINT}/indexers/{indexer_name}/run?api-version={API_VERSION}'
    resp = requests.post(url, headers=get_search_headers())
    if resp.status_code == 202:
        print(f'  ▶ {indexer_name} 실행 시작')
        return True
    else:
        print(f'  ✗ {indexer_name} 실행 실패: {resp.status_code} {resp.text[:200]}')
        return False


def get_indexer_status(indexer_name: str) -> dict:
    """인덱서 상태 조회"""
    url = f'{SEARCH_ENDPOINT}/indexers/{indexer_name}/status?api-version={API_VERSION}'
    resp = requests.get(url, headers=get_search_headers())
    return resp.json()


def wait_indexer(indexer_name: str, timeout_sec: int = 1200, poll_interval: int = 15):
    """인덱서 완료 대기"""
    start = time.time()
    print(f'  ⏳ {indexer_name} 완료 대기 중...')
    while True:
        status = get_indexer_status(indexer_name)
        last = status.get('lastResult') or {}
        state = last.get('status', 'unknown')
        processed = last.get('itemsProcessed', 0)
        failed = last.get('itemsFailed', 0)
        
        elapsed = int(time.time() - start)
        print(f'    [{elapsed:>4d}s] status={state}, processed={processed}, failed={failed}')
        
        if state in ('success', 'transientFailure', 'persistentFailure'):
            return state, last
        if elapsed > timeout_sec:
            print(f'    ⚠ timeout ({timeout_sec}s)')
            return 'timeout', last
        time.sleep(poll_interval)

In [10]:
# ── [B-PDF] Basic PDF 인덱서 실행 ──────────────────────────────
print('='*60)
print('[B-PDF] Basic PDF 인덱서 실행')
print('    DI Layout → markdown_split → Embedding')
print('='*60)

if run_indexer(PDF_INDEXER):
    time.sleep(5)
    state, result = wait_indexer(PDF_INDEXER)
    print(f'\n  결과: {state}')
    if result:
        print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
        if result.get('errors'):
            for err in result['errors'][:3]:
                print(f'    ERROR: {err.get("message", "")[:200]}')

# ── [B-PPTX] Basic PPTX 인덱서 실행 ────────────────────────────
print()
print('='*60)
print('[B-PPTX] Basic PPTX 인덱서 실행')
print('    DI Layout → pptx_page_split → Embedding')
print('='*60)

if run_indexer(PPTX_INDEXER):
    time.sleep(5)
    state, result = wait_indexer(PPTX_INDEXER)
    print(f'\n  결과: {state}')
    if result:
        print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
        if result.get('errors'):
            for err in result['errors'][:3]:
                print(f'    ERROR: {err.get("message", "")[:200]}')


[B-PDF] Basic PDF 인덱서 실행
    DI Layout → markdown_split → Embedding
  ▶ st-multimodal-pdf-indexer 실행 시작
  ⏳ st-multimodal-pdf-indexer 완료 대기 중...
    [   0s] status=inProgress, processed=0, failed=0
    [  15s] status=success, processed=3, failed=0

  결과: success
  처리: 3개, 실패: 0개

[B-PPTX] Basic PPTX 인덱서 실행
    DI Layout → pptx_page_split → Embedding
  ▶ st-multimodal-pptx-indexer 실행 시작
  ⏳ st-multimodal-pptx-indexer 완료 대기 중...
    [   0s] status=transientFailure, processed=0, failed=0

  결과: transientFailure
  처리: 0개, 실패: 0개


In [11]:
# ── [C] Verbalized 인덱서 실행 ─────────────────────────────────
print('='*60)
print('[C] Verbalized 인덱서 실행')
print('    DI Layout → GPT-5.4 Verbalize → Markdown Split → Embedding')
print('='*60)

if run_indexer(VERBALIZED_INDEXER):
    time.sleep(5)
    state, result = wait_indexer(VERBALIZED_INDEXER, timeout_sec=1800, poll_interval=20)
    print(f'\n  결과: {state}')
    if result:
        print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
        if result.get('errors'):
            for err in result['errors'][:3]:
                print(f'    ERROR: {err.get("message", "")[:200]}')

[C] Verbalized 인덱서 실행
    DI Layout → GPT-5.4 Verbalize → Markdown Split → Embedding
  ▶ st-multimodal-verbalized-indexer 실행 시작
  ⏳ st-multimodal-verbalized-indexer 완료 대기 중...
    [   0s] status=inProgress, processed=0, failed=0
    [  20s] status=success, processed=3, failed=0

  결과: success
  처리: 3개, 실패: 0개


## 5. 인덱싱 결과 확인

두 인덱스에 저장된 문서 수와 통계를 비교합니다.

In [15]:
def get_index_stats(index_name: str) -> dict:
    url = f'{SEARCH_ENDPOINT}/indexes/{index_name}/stats?api-version={API_VERSION}'
    resp = requests.get(url, headers=get_search_headers())
    return resp.json() if resp.status_code == 200 else {}

print(f'{"인덱스":<40} {"문서 수":>10} {"저장 크기":>15}')
print('-'*70)

for idx_name in [PDF_INDEX, PPTX_INDEX, VERBALIZED_INDEX]:
    stats = get_index_stats(idx_name)
    doc_count = stats.get('documentCount', 'N/A')
    storage = stats.get('storageSize', 0)
    storage_str = f'{storage:,} bytes' if isinstance(storage, int) else 'N/A'
    print(f'{idx_name:<40} {doc_count:>10} {storage_str:>15}')


인덱스                                            문서 수           저장 크기
----------------------------------------------------------------------
st-multimodal-pdf-index                          70 4,121,356 bytes
st-multimodal-pptx-index                          2   194,856 bytes
st-multimodal-verbalized-index                   70 3,845,284 bytes


In [13]:
# ── 인덱서 상세 상태 (최근 실행) ─────────────────────────────
for indexer_name in [PDF_INDEXER, PPTX_INDEXER, VERBALIZED_INDEXER]:
    status = get_indexer_status(indexer_name)
    last = status.get('lastResult') or {}
    print(f'\n{indexer_name}:')
    print(f'  상태: {last.get("status", "N/A")}')
    print(f'  처리: {last.get("itemsProcessed", 0)}개')
    print(f'  실패: {last.get("itemsFailed", 0)}개')
    print(f'  시작: {last.get("startTime", "N/A")}')
    print(f'  종료: {last.get("endTime", "N/A")}')
    if last.get('errors'):
        print(f'  오류:')
        for err in last['errors'][:5]:
            print(f'    - {err.get("message", "")[:150]}')



st-multimodal-pdf-indexer:
  상태: success
  처리: 3개
  실패: 0개
  시작: 2026-05-15T02:08:15.512Z
  종료: 2026-05-15T02:08:31.964Z

st-multimodal-pptx-indexer:
  상태: success
  처리: 2개
  실패: 0개
  시작: 2026-05-15T02:13:33.893Z
  종료: 2026-05-15T02:13:40.639Z

st-multimodal-verbalized-indexer:
  상태: success
  처리: 3개
  실패: 0개
  시작: 2026-05-15T02:13:56.846Z
  종료: 2026-05-15T02:14:17.046Z


## 6. 샘플 문서 확인 (Quick Look)

각 인덱스에서 첫 3개 문서를 조회하여 인덱싱 품질을 빠르게 확인합니다.

In [16]:
def search_sample(index_name: str, top: int = 3) -> list:
    """인덱스에서 샘플 문서 조회"""
    url = f'{SEARCH_ENDPOINT}/indexes/{index_name}/docs/search?api-version={API_VERSION}'
    payload = {
        'search': '*',
        'top': top,
        'select': 'id,source_file_name,content_type,content',
    }
    resp = requests.post(url, headers=get_search_headers(), json=payload)
    if resp.status_code == 200:
        return resp.json().get('value', [])
    return []

for label, idx in [('B-PDF', PDF_INDEX), ('B-PPTX', PPTX_INDEX), ('C-Verbalized', VERBALIZED_INDEX)]:
    print('='*60)
    print(f'[{label}] {idx}')
    print('='*60)
    for doc in search_sample(idx):
        print(f'\n  📄 {doc.get("source_file_name", "?")} [{doc.get("content_type", "?")}]')
        content = doc.get('content', '') or ''
        print(f'     {content[:200]}...' if len(content) > 200 else f'     {content}')
    print()


[B-PDF] st-multimodal-pdf-index

  📄 ST_0008_0001055.pdf [None]
     d>
<td>-2</td>
<td>1</td>
</tr>
<tr>
<td>33</td>
<td>-1</td>
<td>1</td>
<td>-2</td>
<td>2</td>
</tr>
<tr>
<td>34</td>
<td>-3</td>
<td>-3</td>
<td>-3</td>
<td>-3</td>
</tr>
</table>

...

  📄 ST_0008_0001055.pdf [None]
     003). Approaches to field research. 14-32.

Czarniawska, B. (2014). Social science research: From field to desk.

Ellingsen, I., Størksen, I., & Stephens. P. (2010). Q methodology in social work
...

  📄 ST_0008_0001104.pdf [None]
     행복에 영향을 미치는 여가를 다차원적인 측면에서 해석하고 연구하는 것이 필요
하다고 판단되어 개인의 여가선택 및 긍정적 여가활동 참여를 위해 여가 리터러시에 대해
살펴보는데 그 목적이 있다.

연구목적을 달성하기 위해 설정한 연구문제는 다음과 같다.

첫째, 여가 리터러시의 개념은 무엇인가?

둘째, 여가 리터러시의 구성 요인은 무엇인가?

...

[B-PPTX] st-multimodal-pptx-index

  📄 ST_0008_0001055.pptx [None]
     <!-- Slide 1 -->
예비 사회과 교사의 현지조사에 대한 인식 유형 탐색: Q방법론적 접근

Exploring the Perception Types of Pre-Service Social Studies Teachers

on the Fieldwork: Q-Methodological Approach

곽민정(인하대학교 다문화교육 박사과정)...

  📄 

---

## 정리

| 단계 | 설명 |
|------|------|
| PDF/PPTX 업로드 | Blob Storage `raw-documents/raw/pdf/st/` 에 업로드 |
| [B] Basic 인덱서 실행 | DI Layout → SplitSkill(markdown) → Embedding |
| [C] Verbalized 인덱서 실행 | DI Layout → GPT-5.4 Verbalize → Split → Embedding |
| 결과 확인 | 문서 수, 청크 품질 비교 |

### 다음 단계
→ [06-multimodal-search.ipynb](06-multimodal-search.ipynb) — 두 인덱스의 검색 품질 비교 및 RAG 실험